In [ ]:
import pandas as pd
import numpy as np
from Association_Rule_Mining import create_continous_dataset
from scipy.stats import t as t_dist

In [ ]:
def calc_ema_volatility(df, span=7):
    prev_day_start = df.close.index.searchsorted(df.close.index - pd.Timedelta(days=1))
    prev_day_start = prev_day_start[prev_day_start > 0]
    prev_day_start = pd.Series(df.close.index[prev_day_start - 1], index=df.close.index[df.close.shape[0] - prev_day_start.shape[0]:])
    daily_returns = df.close.loc[prev_day_start.index] / df.close.loc[prev_day_start.values].values - 1
    vol = daily_returns.ewm(span=span).std()

    rolling_mean = vol.rolling(window=span).mean()
    return rolling_mean.mean()

In [ ]:
def calc_max_drawdown(df):
    df['return'] = df.close.pct_change()
    df['cum'] = (1 + df['return']).cumprod()
    running_max = df['cum'].cummax()
    mdd = ((df['cum'] - running_max) / running_max).min()
    return mdd

In [ ]:
def compare_drawdowns_and_volatilities(starting_date, ending_date):
    continous_df_btc = create_continous_dataset('BTC-USD', starting_date=starting_date, ending_date=ending_date)
    print(f"BTC-USD Volatility: {calc_ema_volatility(continous_df_btc)}")
    print(f"BTC-USD Max-Drawdown: {calc_max_drawdown(continous_df_btc)}")
    continous_df_msft = create_continous_dataset('MSFT', starting_date=starting_date, ending_date=ending_date)
    print(f"MSFT Volatility: {calc_ema_volatility(continous_df_msft)}")
    print(f"MSFT Max-Drawdown: {calc_max_drawdown(continous_df_msft)}")
    continous_df_amzn = create_continous_dataset('AMZN', starting_date=starting_date, ending_date=ending_date)
    print(f"AMZN Volatility: {calc_ema_volatility(continous_df_amzn)}")
    print(f"AMZN Max-Drawdown: {calc_max_drawdown(continous_df_amzn)}")
# compare_drawdowns_and_volatilities()

In [ ]:
def evaluate_buy_and_hold(market_data):
    df = market_data.copy()
    df["Return"] = df["close"].pct_change()
    df = df.dropna()

    # Buy once, hold all the way
    cumulative_return = (1 + df["Return"]).prod() - 1

    # Win rate = fraction of days with positive daily return
    win_rate = (df["Return"] > 0).mean() * 100

    return cumulative_return * 100, win_rate

In [ ]:
def evaluate_sma_crossover(market_data, short_window=10, long_window=30):
    df = market_data.copy()
    df["SMA_Short"] = df["close"].rolling(window=short_window).mean()
    df["SMA_Long"] = df["close"].rolling(window=long_window).mean()
    df = df.dropna()

    # Position: 1 = long, -1 = short (you can also use 0 for flat if you prefer)
    df["Position"] = np.where(df["SMA_Short"] > df["SMA_Long"], 1, -1)

    # Daily returns
    df["Return"] = df["close"].pct_change()
    df["Strategy_Return"] = df["Position"].shift(1) * df["Return"]

    # Metrics
    cumulative_return = (1 + df["Strategy_Return"]).prod() - 1
    win_rate = (df["Strategy_Return"] > 0).mean() * 100

    return cumulative_return * 100, win_rate

In [ ]:
def optimize_crossover(dataset):
    shorts = list(range(5,20))
    longs = list(range(20,50))

    best = 0
    short_wind = 0
    long_wind = 0

    for s in shorts:
        for l in longs:
            win_rate = evaluate_sma_crossover(pd.DataFrame(dataset, columns=["close"]), s, l)[1]
            if win_rate > best:
                best = win_rate
                short_wind = s
                long_wind = l
    print(best, short_wind, long_wind)
    
# btc_df = create_continous_dataset('BTC-USD', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
# optimize_crossover(btc_df)
# print(evaluate_buy_and_hold(btc_df))

# amzn_df = create_continous_dataset('AMZN', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
# optimize_crossover(amzn_df)
# print(evaluate_buy_and_hold(amzn_df))

# msft_df = create_continous_dataset('MSFT', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
# optimize_crossover(msft_df)
# print(evaluate_buy_and_hold(msft_df))

In [ ]:
def welch_t_test(mean1, std1, n1, mean2, std2, n2):
    # Welch t-statistic
    numerator = mean1 - mean2
    denominator = sqrt((std1**2)/n1 + (std2**2)/n2)
    t_stat = numerator / denominator

    # Degrees of freedom
    df_num = ((std1**2)/n1 + (std2**2)/n2)**2
    df_den = ((std1**2)**2) / (n1**2 * (n1-1)) + ((std2**2)**2) / (n2**2 * (n2-1))
    df = df_num / df_den

    # Two-tailed p-value
    p_value = 2 * (1 - t_dist.cdf(abs(t_stat), df))

    return t_stat, p_value

In [ ]:
# #2018-2019
# #AMZN
# print("***AMZN***")
# print(welch_t_test(63.64, 3.21, 5, 66.21, 6.22, 5)) #Sentiment Only
# print(welch_t_test(64.55, 2.03, 5, 66.21, 6.22, 5)) #Technical Only
# print(welch_t_test(62.42, 3.28, 5, 66.21, 6.22, 5)) #TP High lift Only
# print(welch_t_test(65.81, 1.33, 5, 66.21, 6.22, 5)) #TP Low lift Only
# print(welch_t_test(57.45, 6.46, 5, 66.21, 6.22, 5)) #TP Zero lift Only
# #BTC-USD
# print("***BTC-USD***")
# print(welch_t_test(54.55, 0, 5, 66.15, 0.60, 5)) #Sentiment Only
# print(welch_t_test(55.35, 2.57, 5, 66.15, 0.60, 5)) #Technical Only
# print(welch_t_test(57.22, 4.66, 5, 66.15, 0.60, 5)) #TP High lift Only
# print(welch_t_test(51.14, 3.42, 5, 66.15, 0.60, 5)) #TP Low lift Only
# print(welch_t_test(60.27, 3.92, 5, 66.15, 0.60, 5)) #All Parameters
# #MSFT
# print("***MSFT***")
# print(welch_t_test(70, 0, 5, 75, 7.14, 5)) #Sentiment Only
# print(welch_t_test(62.50, 15, 5, 75, 7.14, 5)) #Technical Only
# print(welch_t_test(71.43, 3.89, 5, 75, 7.14, 5)) #TP High lift Only
# print(welch_t_test(64.77, 2.27, 5, 75, 7.14, 5)) #TP Zero lift Only
# print(welch_t_test(72.62, 5.99, 5, 75, 7.14, 5)) #All Parameters

In [ ]:
def test_and_rel_performance(mapped_df, baseline_performance):
    max_values = mapped_df.groupby("Market")["Result Mean"].max()
    max_stds = mapped_df.groupby("Market")["Result Std"].max()
    mapped_df["t-statistic"] = mapped_df.apply(lambda x: float(welch_t_test(x["Result Mean"], x["Result Std"], 5, max_values[x["Market"]], max_stds[x["Market"]], 5)[0]), axis=1)
    mapped_df["p-value"] = mapped_df.apply(lambda x: float(welch_t_test(x["Result Mean"], x["Result Std"], 5, max_values[x["Market"]], max_stds[x["Market"]], 5)[1]), axis=1)
    mapped_df["relative_result"] = (mapped_df["Result Mean"] / baseline_performance) * 100
    mapped_df["relative_std"] = mapped_df["Result Std"] * (100 / baseline_performance)
    return mapped_df

In [ ]:
def print_full_results(market, baseline_performance):
    cases = pd.read_csv("TestResults/DDQN_Test_Results.csv")
    mapped_df = pd.DataFrame()
    pd.options.display.float_format = "{:.3f}".format
    cases = cases[~cases["TestCaseName"].str.contains("LASSO", na=False) & ~cases["TestCaseName"].str.contains("PCA", na=False)]
    cases = cases[cases["Market"] == market]
    mapped_df["Name"] = cases["TestCaseName"]
    mapped_df["Market"] = cases["Market"]
    mapped_df["Result Mean"] = cases["Positive Rewards"].apply(lambda x: float(x.split(",")[0].split(" ")[1]))
    mapped_df["Result Std"] = cases["Positive Rewards"].apply(lambda x: float(x.split(",")[1].split(" ")[2]))
    mapped_df = test_and_rel_performance(mapped_df, baseline_performance)
    return mapped_df.sort_values(by="Result Mean", ascending=False)

In [ ]:
# amzn_shorter_period_values = [
#     {
#         "Name": "AMZN_Sentiment_Only",
#         "Market": "AMZN",
#         "Result Mean": 63.64,
#         "Result Std":  3.21
#     },
#     {
#         "Name": "AMZN_Technical_Only",
#         "Market": "AMZN",
#         "Result Mean": 64.55,
#         "Result Std":  2.03
#     },
#     {
#         "Name": "AMZN_TargetProfitable_Zero_Lift",
#         "Market": "AMZN",
#         "Result Mean": 57.45,
#         "Result Std": 6.46
#     },
#     {
#         "Name": "AMZN_TargetProfitable_Low_Lift",
#         "Market": "AMZN",
#         "Result Mean": 65.81,
#         "Result Std":  1.33
#     },
#     {
#         "Name": "AMZN_TargetProfitable_High_Lift",
#         "Market": "AMZN",
#         "Result Mean": 62.42,
#         "Result Std":  3.28
#     },
#     {
#         "Name": "AZMN_All_Params",
#         "Market": "AMZN",
#         "Result Mean": 66.21,
#         "Result Std":  6.22
#     },
#     {
#         "Name": "AMZN_SMA-crossover",
#         "Market": "AMZN",
#         "Result Mean": 60.22,
#         "Result Std":  0.00
#     },
# ]

# btcusd_shorter_period_values = [
#     {
#         "Name": "BTC_Sentiment_Only",
#         "Market": "BTC-USD",
#         "Result Mean": 54.55,
#         "Result Std":  0.00
#     },
#     {
#         "Name": "BTC_Technical_Only",
#         "Market": "BTC-USD",
#         "Result Mean": 55.35,
#         "Result Std":  2.57
#     },
#     {
#         "Name": "BTC_TargetProfitable_Zero_Lift",
#         "Market": "BTC-USD",
#         "Result Mean": 66.15,
#         "Result Std": 0.60
#     },
#     {
#         "Name": "BTC_TargetProfitable_Low_Lift",
#         "Market": "BTC-USD",
#         "Result Mean": 51.14,
#         "Result Std":  3.42
#     },
#     {
#         "Name": "BTC_TargetProfitable_High_Lift",
#         "Market": "BTC-USD",
#         "Result Mean": 57.22,
#         "Result Std":  4.66
#     },
#     {
#         "Name": "BTC_All_Params",
#         "Market": "BTC-USD",
#         "Result Mean": 60.27,
#         "Result Std":  3.92
#     },
#     {
#         "Name": "BTC_SMA-crossover",
#         "Market": "BTC-USD",
#         "Result Mean": 51.35,
#         "Result Std":  0.00
#     },
# ]

# msft_shorter_period_values = [
#     {
#         "Name": "MSFT_Sentiment_Only",
#         "Market": "MSFT",
#         "Result Mean": 70.00,
#         "Result Std":  0.00
#     },
#     {
#         "Name": "MSFT_Technical_Only",
#         "Market": "MSFT",
#         "Result Mean": 62.50,
#         "Result Std":  15
#     },
#     {
#         "Name": "MSFT_TargetProfitable_Zero_Lift",
#         "Market": "MSFT",
#         "Result Mean": 64.77,
#         "Result Std": 2.27
#     },
#     {
#         "Name": "MSFT_TargetProfitable_Low_Lift",
#         "Market": "MSFT",
#         "Result Mean": 75,
#         "Result Std":  7.14
#     },
#     {
#         "Name": "MSFT_TargetProfitable_High_Lift",
#         "Market": "MSFT",
#         "Result Mean": 71.43,
#         "Result Std":  3.89
#     },
#     {
#         "Name": "MSFT_All_Params",
#         "Market": "MSFT",
#         "Result Mean": 72.62,
#         "Result Std":  5.99
#     },
#     {
#         "Name": "MSFT_SMA-crossover",
#         "Market": "MSFT",
#         "Result Mean": 55.69,
#         "Result Std":  0.00
#     },
# ]

In [ ]:
# #BTC-USD 2018-2019 Full Results
# test_and_rel_performance(pd.DataFrame(btcusd_shorter_period_values), 50.79365079365079).sort_values(by="Result Mean", ascending=False)
# #BTC-USD 2018-2021 Full Results
# print_full_results("BTC-USD", 53.25047801147228)
# #AMZN 2018-2019 Full Results
# test_and_rel_performance(pd.DataFrame(amzn_shorter_period_values), 53.73134328358209).sort_values(by="Result Mean", ascending=False)
# #AMZN 2018-2021 Full Results
# print_full_results("AMZN", 54.39093484419264)
# #MSFT 2018-2019 Full Results
# test_and_rel_performance(pd.DataFrame(msft_shorter_period_values), 54.72636815920397).sort_values(by="Result Mean", ascending=False)
# #MSFT 2018-2021 Full Results
# print_full_results("MSFT", 56.798866855524075)